In [18]:
import sys

!{sys.executable} -m pip install google-cloud-bigquery google-cloud-bigquery-storage pandas pyarrow db-dtypes

In [19]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza z punktu 1.4.

client = bigquery.Client(credentials=credentials, project=credentials.project_id)


In [20]:
query = """
SELECT *
FROM `bigquery-public-data.noaa_gsod.gsod2020`
    limit 10
"""

query_job = client.query(query)
query_result = query_job.result()
df = query_result.to_dataframe()

In [56]:
df

,usaf,wban,name,country,lat,lon,elev
0,007018,None,WXPOD 7018,None,0.000,0.000,+7018.0
1,007026,None,WXPOD 7026,AF,0.000,0.000,+7026.0
2,007070,None,WXPOD 7070,AF,0.000,0.000,+7070.0
3,008268,None,WXPOD8278,AF,32.950,65.567,+1156.7
4,008307,None,WXPOD 8318,AF,0.000,0.000,+8318.0
...,...,...,...,...,...,...,...
29585,A06884,00416,LURAY CAVERNS AIRPORT,US,38.667,-78.501,+0275.2
29586,A07086,00468,CARL R KELLER FIELD AIRPORT,US,41.516,-82.869,+0179.8
29587,A51255,00445,DEMOPOLIS MUNICIPAL AIRPORT,US,32.464,-87.954,+0034.1
29588,A51256,00451,BRANSON WEST MUNICIPAL EMERSO,US,36.699,-93.402,+0411.2


# Misja dodatkowa 3

### 3.1. Sprawdź, ile jest zapisanych wierszy z danymi.

In [22]:
query3_1 = """
select count(*) as count_rows from(
    SELECT  *
    FROM `bigquery-public-data.noaa_gsod.gsod2020`
    UNION ALL
    SELECT * FROM `bigquery-public-data.noaa_gsod.gsod2021`)
"""

df_3_1 = client.query(query3_1).to_dataframe()
df_3_1['count_rows'].loc[0]


np.int64(8145755)

### 3.2. Sprawdź, ile krajów jest uwzględnionych w danych.


In [51]:
query3_2 = """
        select count (Distinct country) as country_count from (SELECT *
        FROM `bigquery-public-data.noaa_gsod.gsod2020` AS gsod
                 INNER JOIN `bigquery-public-data.noaa_gsod.stations` AS st
                            ON st.usaf = gsod.stn and gsod.wban = st.wban
        UNION ALL
        SELECT *
        FROM `bigquery-public-data.noaa_gsod.gsod2021` AS gsod
                 INNER JOIN `bigquery-public-data.noaa_gsod.stations` AS st
                            ON st.usaf = gsod.stn and gsod.wban = st.wban )

        """

df_3_2= client.query(query3_2).to_dataframe()
df_3_2['country_count'].loc[0]


np.int64(239)

### 3.3. Sprawdź, w jaki sposób zapisywane są dzienne informacje dla poszczególnych lokalizacji.


In [24]:
query = """
SELECT *
FROM `bigquery-public-data.noaa_gsod.gsod2020`
    limit 10
"""

query_job = client.query(query)
query_result = query_job.result()
df = query_result.to_dataframe()
df

,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,...,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,010030,99999,2020-10-12,2020,10,12,27.0,4,20.5,4,...,None,0.0,I,999.9,0,0,0,0,0,0
1,010350,99999,2020-11-08,2020,11,08,27.9,4,9999.9,0,...,None,0.0,I,999.9,0,0,0,0,0,0
2,010370,99999,2020-12-19,2020,12,19,22.3,4,16.6,4,...,None,0.0,I,999.9,0,0,0,0,0,0
3,010420,99999,2020-03-03,2020,03,03,21.5,4,9999.9,0,...,None,0.0,I,999.9,0,0,0,0,0,0
4,010420,99999,2020-02-18,2020,02,18,23.9,4,9999.9,0,...,*,0.0,I,999.9,0,0,0,0,0,0
5,010430,99999,2020-01-15,2020,01,15,29.2,4,9999.9,0,...,None,0.0,I,999.9,0,0,0,0,0,0
6,010430,99999,2020-02-01,2020,02,01,18.9,4,9999.9,0,...,*,0.0,I,999.9,0,0,0,0,0,0
7,010430,99999,2020-01-17,2020,01,17,25.1,4,9999.9,0,...,*,0.0,I,999.9,0,0,0,0,0,0
8,010430,99999,2020-03-18,2020,03,18,29.5,4,9999.9,0,...,None,0.0,I,999.9,0,0,0,0,0,0
9,010430,99999,2020-04-03,2020,04,03,24.0,4,9999.9,0,...,None,0.0,I,999.9,0,0,0,0,0,0


 0.   stn - numer identyfikacyjny stacji
 1.   wban - numer Weather Bureau Air Force Navy, używany historycznie przez amerykańskie biura pogodowe oraz wojsko
 2.   date - data obserwacji
 3.   year - rok obserwacji
 4.   mo   - miesiąc obserwacji
 5.   da   - dzień obserwacji
 6.   temp - średnia temperatura dobowa w stopniach Fahrenheita, zaokrąglona do części dziesiętnych *
 7.   count_temp  - liczba obserwacji
 8.   dewp - średni, dobowy punkt rosy w stopniach Fahrenheita, zaokrąglony do części dziesiętnych *
 9.   count_dewp  - liczba obserwacji
 10.  slp - średnie ciśnienie nad poziomem morza (mbar), zaokrąglone do części dziesiętnych *
 11.  count_slp - liczba obserwacji
 12.  stp - średnie ciśnienie na poziomie stacji pogodowej (mbar), zaokrąglone do części dziesiętnych *
 13.  count_stp - liczba obserwacji
 14.  visib - średnia widoczność w milach, zaokrąglona do części dziesiętnych **
 15.  count_visib   - liczba obserwacji
 16.  wdsp - średnia predkość wiatru w węzłach, zaokrąglona do części dziesiętnych **
 17.  count_wdsp   - liczba obserwacji
 18.  mxpsd - maksymalna, utrzymująca się prędkość wiatru w węzłach, zaokrąglona do części dziesiętnych **
 19.  gust - maksymalna prędkość podmuchu wiatru w węzłach, zaokrąglona do części dziesiętnych **
 20.  max - maksymalna temperatura zarejestrowana dla danego kraju i regionu w stopniach Fahrenheita, zaokrąglona do części dziesiętnych *
 21.  flag_max - wskaźnik informujący czy maksymalna temperatura została pobrana z raportu, czy wyliczona z danych godzinowych
 22.  min - minimalna temperatura zarejestrowana dla danego kraju i regionu w stopniach Fahrenheita, zaokrąglona do części dziesiętnych *
 23.  flag_min - wskaźnik informujący czy maksymalna temperatura została pobrana z raportu, czy wyliczona z danych godzinowych
 24.  prcp - całkowita suma opadów (również roztopiony śnieg) w calach i setnych częściach milimetra ***
 25.  flag_prcp - wskaźnik informujący ilości raportów zgłoszonych w ciągu dnia
 - A - 1 raport 6-godzinnej ilości opadów
 - B - suma 2 raportów 6-godzinnych
 - C - suma 3 raportów 6-godzinnych
 - D - suma 4 raportów 6-godzinnych
 - E - 1 raport o 12-godzinnej ilości opadów
 - F - suma 2 raportów 12-godzinnych
 - G - 1 raport o 24-godzinnej ilości opadów
 26.  sndp - grbość pokrywy śnieżnej w calach, ostatni pomiar w ciągu dnia, jeżeli wykonano więcej niż 1 **
 27.  fog - wskaźnik informujący o występowaniu mgły w ciągu dnia
 28.  rain_drizzle - wskaźnik informujący o występowaniu mżawki w ciągu dnia
 29.  snow_ice_pellets - wskaźnik informujący o występowaniu krup śnieżnych bądź lodowych (mieszanka śniegu i lodu) w ciągu dnia
 30.  hail - wskaźnik informujący o występowaniu gradu w ciągu dnia
 31.  thunder - wskaźnik informujący o wystąpieniu burzy w ciągu dnia
 32.  tornado_funnel_cloud - wskaźnik informujący o wystąpieniu tornada bądź leja kondensacyjnego w ciągu dnia

\* brakujące wartości są zastąpione przez 9999.9
\** brakujące wartości są zastąpione przez 999.9
\*** brakujące wartości są zastąpione przez 99.99


### 3.4. Sprawdź, w jaki sposób zapisywane są wartości liczbowe.


wartości pomiarów zapisywane są w postaci liczb zmiennoprzecinkowych (float) zaokrąglone do pierwszego miejsca po przecinku, natomiast wartości opisyjące liczbe pomiarów są liczbami całkowitymi (integer)

### 3.5. Sprawdź, jaki przedział czasowy jest uwzględniony w danych. Dodatkowo porównaj zakresy czasowe dla różnych zmiennych pogodowych, takich jak temperatura i opady.


### 3.6. Sprawdź więcej informacji (co najmniej 5 różnych) o danych pogodowych. W tym celu nie wykonuj żadnych dodatkowych obliczeń. Wyniki przedstaw w formie krótkiego raportu.

# Część 4

### Przeanalizuj poniższe przypadki. Zastanów się, jakie dane potrzebujesz do każdego z nich, a następnie zapisz je w osobnych, jak najprostszych obiektach DataFrame. Na tym etapie nie wykonuj analiz ani obliczeń statystycznych. Wykonaj jedynie podstawowe uporządkowanie danych: wybierz potrzebne kolumny, zadbaj o poprawne typy danych (np. data), usuń duplikaty oraz jednoznacznie oznacz brakujące wartości. Gotowe dane z obiektów DataFrame zapisz w osobnych plikach CSV.

### 4.1. Chcemy posiadać podstawowe informacje o lokalizacjach pomiarów pogodowych (stacje) oraz krajach, tak aby dane były zrozumiałe dla człowieka i możliwe do dalszego przetwarzania.



In [47]:
query = """
        SELECT usaf, wban, name, country, lat, lon, elev
        FROM `bigquery-public-data.noaa_gsod.stations`
        """

df = client.query(query).to_dataframe()
df['wban'] = df['wban'].replace('99999', None)
df = df.drop_duplicates()
df.to_csv('data/zad4_1.csv',index=False)


### 4.2. Chcemy wygenerować podstawowe zestawienia dotyczące warunków pogodowych na świecie (np. temperatura, opady, wiatr) w ujęciu dziennym.



In [44]:
query = """
        SELECT stn, wban, temp, dewp, slp, stp, visib, wdsp, prcp, sndp, `date`
        FROM `bigquery-public-data.noaa_gsod.gsod2020`
        """

df = client.query(query).to_dataframe()
df['wban'] = df['wban'].replace('99999', None)
df['temp'] = df['temp'].replace(9999.9, None).astype(np.float64)
df['dewp'] = df['dewp'].replace(9999.9, None).astype(np.float64)
df['slp'] = df['slp'].replace(9999.9, None).astype(np.float64)
df['stp'] = df['stp'].replace(9999.9, None).astype(np.float64)
df['visib'] = df['visib'].replace(999.9, None).astype(np.float64)
df['wdsp'] = df['wdsp'].replace('999.9', None).astype(np.float64)
df['prcp'] = df['prcp'].replace(99.99, None).astype(np.float64)
df['sndp'] = df['sndp'].replace(999.9, None).astype(np.float64)
df = df.drop_duplicates()
df.to_csv('data/zad4_2.csv',index=False)
df

,stn,wban,temp,dewp,slp,stp,visib,wdsp,prcp,sndp,date
0,012120,None,49.5,46.9,997.6,995.8,NaN,10.0,0.00,NaN,2020-09-22
1,013520,None,25.3,NaN,NaN,999.9,NaN,23.3,0.03,74.0,2020-02-22
2,013590,None,21.5,20.4,1014.9,920.6,NaN,1.7,0.00,NaN,2020-12-16
3,013760,None,38.5,37.7,NaN,999.9,NaN,4.7,0.02,NaN,2020-10-30
4,013760,None,55.6,34.3,NaN,999.9,NaN,6.6,0.00,NaN,2020-09-10
...,...,...,...,...,...,...,...,...,...,...,...
4119200,A51255,00445,79.4,74.0,NaN,10.0,10.0,1.7,0.00,NaN,2020-09-13
4119201,A51255,00445,62.6,42.6,NaN,10.7,8.5,5.8,0.00,NaN,2020-04-10
4119202,A51256,00451,78.0,72.2,NaN,960.5,9.7,7.6,NaN,NaN,2020-08-28
4119203,A51256,00451,80.9,72.6,NaN,970.8,10.0,3.2,0.00,NaN,2020-07-26


### 4.3. Chcemy poznać zjawiska ekstremalne w danych pogodowych poprzez uwypuklenie skrajnych wartości (np. bardzo wysokie/niskie temperatury, intensywne opady, silny wiatr).



In [27]:
query = """
        SELECT stn, wban, mxpsd, gust, max, min, prcp, snow_ice_pellets, hail, thunder, tornado_funnel_cloud, `date`
        FROM `bigquery-public-data.noaa_gsod.gsod2020`
        """

df = client.query(query).to_dataframe()
df['wban'] = df['wban'].replace('99999', None)
df['mxpsd'] = df['mxpsd'].replace('999.9', None).astype(np.float64)
df['gust'] = df['gust'].replace(999.9, None).astype(np.float64)
df['prcp'] = df['prcp'].replace(99.99, None).astype(np.float64)
df['snow_ice_pellets'] = df['snow_ice_pellets'].astype(np.int32).astype(np.bool)
df['hail'] = df['hail'].astype(np.int32).astype(np.bool)
df['thunder'] = df['thunder'].astype(np.int32).astype(np.bool)
df['tornado_funnel_cloud'] = df['tornado_funnel_cloud'].astype(np.int32).astype(np.bool)
df = df.drop_duplicates()

df.to_csv('data/zad4_3.csv',index=False)
df

,stn,wban,mxpsd,gust,max,min,prcp,snow_ice_pellets,hail,thunder,tornado_funnel_cloud,date
0,010420,None,25.8,34.0,33.4,30.0,0.0,False,False,False,False,2020-10-13
1,010430,None,27.2,NaN,26.8,20.7,0.0,False,False,False,False,2020-02-05
2,011020,None,NaN,NaN,48.6,41.9,0.0,False,False,False,False,2020-11-04
3,011070,None,22.0,NaN,44.6,42.8,NaN,False,False,False,False,2020-11-22
4,011990,None,9.7,12.2,39.2,37.9,0.0,False,False,False,False,2020-10-05
...,...,...,...,...,...,...,...,...,...,...,...,...
4119200,A51256,00451,6.0,NaN,59.0,32.0,0.0,False,False,False,False,2020-10-30
4119201,A51256,00451,15.0,42.9,84.2,62.6,NaN,False,False,True,False,2020-06-04
4119202,A51256,00451,8.9,15.9,84.2,60.8,NaN,False,False,False,False,2020-09-15
4119203,A51256,00451,8.9,NaN,50.0,32.0,0.0,False,False,False,False,2020-01-28


### 4.4. Chcemy przygotować dane umożliwiające analizę zmian warunków pogodowych w czasie (np. dla wybranych lokalizacji lub zmiennych).



In [28]:
query = """
        SELECT stn, wban, temp, dewp, stp, visib, wdsp, prcp, `date`
        FROM `bigquery-public-data.noaa_gsod.gsod2020`
        """

df = client.query(query).to_dataframe()
df['wban'] = df['wban'].replace('99999', None)
df['temp'] = df['temp'].replace(9999.9, None).astype(np.float64)
df['dewp'] = df['dewp'].replace(9999.9, None).astype(np.float64)
df['stp'] = df['stp'].replace(9999.9, None).astype(np.float64)
df['visib'] = df['visib'].replace(999.9, None).astype(np.float64)
df['wdsp'] = df['wdsp'].replace('999.9', None).astype(np.float64)
df['prcp'] = df['prcp'].replace(99.99, None).astype(np.float64)
df = df.drop_duplicates()

df.to_csv('data/zad4_4.csv',index=False)
df

,stn,wban,temp,dewp,stp,visib,wdsp,prcp,date
0,010430,None,23.6,NaN,999.9,NaN,13.6,0.00,2020-02-26
1,011070,None,54.5,48.2,999.9,6.2,17.0,0.00,2020-10-04
2,011440,None,39.8,31.4,999.9,NaN,12.1,0.00,2020-12-09
3,013450,None,44.4,41.3,999.9,NaN,16.6,0.00,2020-09-21
4,013520,None,22.4,NaN,999.9,NaN,17.5,0.00,2020-02-20
...,...,...,...,...,...,...,...,...,...
4119200,999999,92827,77.9,NaN,999.9,NaN,NaN,0.01,2020-09-08
4119201,999999,93245,54.5,NaN,999.9,NaN,NaN,0.00,2020-04-18
4119202,A00021,63884,74.6,72.0,953.3,6.6,NaN,NaN,2020-09-02
4119203,A00021,63884,44.9,33.2,947.5,9.9,NaN,NaN,2020-02-02


### 4.5. Zdefiniuj własny dodatkowy przypadek związany z danymi pogodowymi.



#### Dane opisujące zjawiska pogodowe wpływające na poruszanie się samochodem

In [29]:
query = """
        SELECT stn, wban, visib, wdsp, mxpsd, gust, fog, hail, thunder, tornado_funnel_cloud, `date`
        FROM `bigquery-public-data.noaa_gsod.gsod2020`
        """

df = client.query(query).to_dataframe()
df['wban'] = df['wban'].replace('99999', None)
df['visib'] = df['visib'].replace(999.9, None).astype(np.float64)
df['wdsp'] = df['wdsp'].replace('999.9', None).astype(np.float64)
df['mxpsd'] = df['mxpsd'].replace('999.9', None).astype(np.float64)
df['gust'] = df['gust'].replace(999.99, None).astype(np.float64)

df['fog'] = df['fog'].astype(np.int32).astype(np.bool)
df['hail'] = df['hail'].astype(np.int32).astype(np.bool)
df['thunder'] = df['thunder'].astype(np.int32).astype(np.bool)
df['tornado_funnel_cloud'] = df['tornado_funnel_cloud'].astype(np.int32).astype(np.bool)
df = df.drop_duplicates()

df.to_csv('data/zad4_5.csv',index=False)
df

,stn,wban,visib,wdsp,mxpsd,gust,fog,hail,thunder,tornado_funnel_cloud,date
0,010020,None,NaN,10.0,11.1,14.6,False,False,False,False,2020-05-11
1,010070,None,NaN,7.5,14.8,18.5,False,False,False,False,2020-05-11
2,010420,None,NaN,17.0,48.6,999.9,False,False,False,False,2020-03-12
3,010920,None,NaN,17.3,19.4,35.4,False,False,False,False,2020-12-25
4,011440,None,NaN,28.6,34.4,45.3,False,False,False,False,2020-11-06
...,...,...,...,...,...,...,...,...,...,...,...
4119200,A07359,00240,10.0,4.3,11.1,17.1,False,False,False,False,2020-04-27
4119201,A51255,00445,9.9,1.9,8.0,999.9,False,False,False,False,2020-06-16
4119202,A51255,00445,10.0,3.7,12.0,15.9,False,False,False,False,2020-05-14
4119203,A51256,00451,10.0,4.8,14.0,22.9,False,False,False,False,2020-04-16


## Część 5

### Połącz ze sobą wszystkie dane otrzymane w części 4. Na ich podstawie utwórz jeden spójny zbiór danych i zapisz go jako oddzielny obiekt DataFrame. Jeżeli uznasz, że zasadne jest utworzenie więcej niż jednego zbioru, wykonaj to i uzasadnij swoją decyzję. Zadbaj o spójność struktury danych, w szczególności zgodność nazw kolumn, poprawność typów danych, brak duplikatów, jednoznaczne oznaczenie brakujących wartości. Wynik zapisz w pliku lub plikach CSV.

In [48]:
localization = pd.read_csv('data/zad4_1.csv')
daily_weather = pd.read_csv('data/zad4_2.csv')
extreme_weather = pd.read_csv('data/zad4_3.csv')
change_weather = pd.read_csv('data/zad4_4.csv')
car_weather = pd.read_csv('data/zad4_5.csv')

C:\Users\dorot\AppData\Local\Temp\ipykernel_12692\2591248430.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  daily_weather = pd.read_csv('data/zad4_2.csv')
C:\Users\dorot\AppData\Local\Temp\ipykernel_12692\2591248430.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  extreame_weather = pd.read_csv('data/zad4_3.csv')


,usaf,wban,name,country,lat,lon,elev
0,007018,NaN,WXPOD 7018,NaN,0.000,0.000,7018.0
1,007026,NaN,WXPOD 7026,AF,0.000,0.000,7026.0
2,007070,NaN,WXPOD 7070,AF,0.000,0.000,7070.0
3,008268,NaN,WXPOD8278,AF,32.950,65.567,1156.7
4,008307,NaN,WXPOD 8318,AF,0.000,0.000,8318.0
...,...,...,...,...,...,...,...
29585,A06884,416.0,LURAY CAVERNS AIRPORT,US,38.667,-78.501,275.2
29586,A07086,468.0,CARL R KELLER FIELD AIRPORT,US,41.516,-82.869,179.8
29587,A51255,445.0,DEMOPOLIS MUNICIPAL AIRPORT,US,32.464,-87.954,34.1
29588,A51256,451.0,BRANSON WEST MUNICIPAL EMERSO,US,36.699,-93.402,411.2


In [55]:
daily_weather.merge(localization, how='inner', left_on= 'stn', right_on='usaf').drop(columns = ['usaf'])
extreme_weather.merge(localization, how='inner', left_on= 'stn', right_on='usaf').drop(columns = ['usaf'])
change_weather.merge(localization, how='inner', left_on= 'stn', right_on='usaf').drop(columns = ['usaf'])
car_weather.merge(localization, how='inner', left_on= 'stn', right_on='usaf').drop(columns = ['usaf'])


MemoryError: Unable to allocate 3.18 GiB for an array with shape (4, 106596483) and data type float64

## Część 6

### Połącz ze sobą dane otrzymane w części 5 oraz dane znajdujące się w dodatkowym pliku CSV zawierającym informacje o produkcji rolniczej.
Wybierz jedno z poniższych źródeł danych:

### 6.1  Dane FAOSTAT (zalecane)

Dane dotyczące produkcji rolniczej dostępne w bazie FAOSTAT: https://www.fao.org/faostat/en/#data/QCL

Dane obejmują m.in.:

Area – nazwa kraju

Item – rodzaj uprawy

Year – rok

Element – typ wartości (np. Production, Yield)

Value – wartość

### 6.2. Crop Yield Prediction Dataset (Kaggle)

Dane dotyczące plonów rolniczych dostępne na platformie Kaggle: https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset

Zbiór zawiera m.in.:

nazwę kraju,
rok,
rodzaj uprawy,
wielkość plonu lub produkcji.
Połącz dane pogodowe z danymi rolniczymi, zwracając szczególną uwagę na zgodność nazw krajów, dopasowanie zakresów czasowych, różnice w strukturze danych (np. dane dzienne vs roczne), brakujące wartości.

Nowy zbiór danych zapisz jako oddzielny obiekt DataFrame. Jeżeli uważasz, że należy stworzyć kilka takich obiektów, zrób to i zapisz swoje uzasadnienie.

Zadbaj o spójność struktury danych, w szczególności na zgodność nazw kolumn, poprawność typów danych, brak duplikatów, jednoznaczne oznaczenie brakujących wartości.

Wynik tego zadania zapisz w pliku lub plikach CSV.